# Stage 1 - score every text with the StressTwin-F ensemble (GPU runtime)

Uses the weights exported by the retrain notebook. Run it twice for the ablation: `SCORER = 'stresstwin_f'`, then `'plain_ft'`.

In [ ]:
# one-time: put the package on Drive and install it. Upload the folder stresstwin_p/ (the one with pyproject.toml)
# to  MyDrive/stresstwin_p  first.
from google.colab import drive; drive.mount('/content/drive')
%pip -q install -e /content/drive/MyDrive/stresstwin_p
import os; WORK='/content/drive/MyDrive/stresstwin_work'; os.makedirs(WORK, exist_ok=True)

In [ ]:
%pip install -q transformers==4.53.2 tokenizers==0.21.4
import numpy as np, pandas as pd
from stresstwin_p.stf_scorer import load_ensemble
from stresstwin_p.score_texts import score_table
EXPORT = f'{WORK}/stf_export'                               # copy stf_export here from the retrain notebook
SCORER = 'stresstwin_f'                                     # 'plain_ft' for the ablation run
ens = load_ensemble(f'{EXPORT}/backbone', f'{EXPORT}/ckpt', name=SCORER)
print('seeds', ens.n_seeds, 'device', ens.device)

In [ ]:
# 100-text smoke test first: confirms logits have the right sign and speed
import pandas as pd, numpy as np
t = pd.read_csv(f'{WORK}/tables/texts.csv')
sc = ens
s = score_table(t.head(100), sc, batch_size=32); print(s.describe().T[['mean','std','min','max']])

In [ ]:
# full run, in chunks so a Colab disconnect loses at most one chunk
import os
CH=20000; os.makedirs(f'{WORK}/scores_{SCORER}', exist_ok=True)
for i in range(0, len(t), CH):
    f=f'{WORK}/scores_{SCORER}/part{i//CH:04d}.csv'
    if os.path.exists(f): continue
    score_table(t.iloc[i:i+CH], ens, batch_size=64).to_csv(f, index=False); print('done', f)

In [ ]:
import glob
pd.concat([pd.read_csv(f) for f in sorted(glob.glob(f'{WORK}/scores_{SCORER}/part*.csv'))]).to_csv(f'{WORK}/scores_{SCORER}.csv', index=False); print('merged')